# 02 · Healthy signal behaviour
**Goal:** understand every observed signal before deciding what a model should receive.

In [ ]:
from pathlib import Path
import os
import sys

# Works when Jupyter starts from the repository root or from nbs.
HELPERS = Path.cwd() if (Path.cwd() / "fleetguard_eda.py").exists() else Path.cwd() / "nbs"
sys.path.insert(0, str(HELPERS))
import fleetguard_eda as eda
import panel as pn
import plotly.express as px

eda.setup()
ROOT = eda.project_root()
SOURCE = Path(os.environ.get("FLEETGUARD_EDA_SOURCE", ROOT / "data/generated/portfolio-v1"))
FEATURES = Path(os.environ.get("FLEETGUARD_EDA_FEATURES", ROOT / "data/features/portfolio-v1"))
SPLIT = "train"  # change to validation to check findings; test is blocked for exploration
ws = eda.Workspace(SOURCE, FEATURES)
pn.pane.Markdown(f"**Source:** `{SOURCE}`  ·  **Features:** `{FEATURES}`  ·  **Explore:** `{SPLIT}`\n\nMaps need internet. Linked controls need a running, trusted notebook kernel.")

In [ ]:
eda.banner("02 · What healthy looks like", "Distributions → operating context → linked relationships → artefacts")

### 1 · Signal availability and spread
Primary reference: healthy-control wagons only. Constants are visible, not discarded silently.

In [ ]:
healthy_profile = eda.quality_profile(ws, SPLIT, variants=["healthy"])
eda.table(healthy_profile)

### 2 · Every numerical signal
Choose a level. Each facet has its own units/scales; density is over valid received samples.

### Known properties
Wheel diameter and handbrake position are metadata, not new measurements each reporting instant.

In [ ]:
eda.static_properties(ws, SPLIT)

In [ ]:
gallery_level = pn.widgets.Select(name="Component level", options=list(eda.NUMERIC))
pn.Column(gallery_level, pn.bind(lambda level: eda.pane(eda.distribution_gallery(ws, level, SPLIT)), gallery_level))

### 3 · States and categories
Counts are received samples, not time occupancy when reports are absent.

In [ ]:
pn.Column(gallery_level, pn.bind(lambda level: eda.state_gallery(ws, level, SPLIT), gallery_level))

### 4 · Inspect a signal, wagon and component
Compare distributions, component spread and chronological behaviour. Empty combinations are explicit.

In [ ]:
explore_level = pn.widgets.Select(name="Explore level", options=list(eda.NUMERIC))
pn.Column(explore_level, pn.bind(lambda level: eda.healthy_explorer(ws, level, SPLIT), explore_level))

### 5 · Linked selection
Box/lasso a region in the relationship plot. The same rows are selected in the time plot. At most 12,000 rows are shown.

In [ ]:
eda.linked_relationship(ws, "axle", SPLIT)

### 6 · Correlation, with engineering context
Spearman correlations pool healthy operating states. Repeated/time-dependent samples are not independent evidence; do not interpret causally.

### Engineering relationships
Inspect pressure response by brake demand and bearing heat above ambient by speed/load.

In [ ]:
eda.engineering_relationships(ws, SPLIT)

In [ ]:
corr_level = pn.widgets.Select(name="Correlation level", options=list(eda.NUMERIC))
clustered = pn.widgets.Checkbox(name="Cluster related signals", value=False)
pn.Column(pn.Row(corr_level, clustered), pn.bind(lambda level, cluster: eda.correlation_view(ws, level, SPLIT, cluster), corr_level, clustered))

### 7 · Geographic context for healthy controls
Report density reflects routes/dwell, not failure risk.

In [ ]:
healthy_wagons = ws.load("wagon", SPLIT, variants=["healthy"])
eda.folium_map(healthy_wagons)

### 8 · Check simulator fingerprints
Inspect constants, exact wheelset/wagon speed relationships, repeated baselines and the power transition. These can be designed simulator relationships, not evidence of real hardware behaviour.

In [ ]:
varying = healthy_profile.assign(Constant=healthy_profile.Distinct <= 1)
eda.pane(eda.figure(px.bar(varying, x="Signal", y="Distinct", color="Level", hover_data=["Unit", "Constant"]), "Distinct valid values — constants and low variation need review"))

### 9 · Record decisions
- Which signals vary with braking/speed/load?
- Which fields are constant or redundant?
- Which relationships are exact by generator design?
- What needs an operating-context feature or careful scaling?

**Next session:** fault evidence, localisation and resolution.